# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading and exploration of the FAIR² dataset, which investigates clinicopathological and molecular features of second primary colorectal cancer in cancer survivors, using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print key metadata
print(f"{metadata.name} (version {metadata.version}): {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Fields with personal sensitive information: {getattr(metadata, 'personalSensitiveInformation', None)}")

## 2. Data Overview
Review available record sets and their IDs, along with associated fields and columns.

The `@id` uniquely identifies every entity (record set, field, column) within the Croissant schema.

In [ ]:
# List all record sets declared in the metadata

record_sets = dataset.record_sets
print("Available record sets in the dataset:")
for rs in record_sets:
    print(f"@id: {rs.id} | name: {rs.name}")
    if hasattr(rs, "fields"):
        print("  Fields:")
        for field in rs.fields:
            field_name = getattr(field, 'name', getattr(field, 'id', None))
            print(f"    @id: {field.id} | name: {field_name} | type: {getattr(field, 'data_type', 'Unknown')}")
        print()

# If record_sets is empty, try to list any record set IDs via metadata directly
if not record_sets:
    print("(No record_set entities found in metadata.record_sets. The dataset may not explicitly declare them, or uses a single main tabular resource.)")
    print("In this case, we'll try to infer the default table from dataset.records().")
    # Check what the dataset yields as record_set IDs:
    records_preview = list(dataset.records())
    if records_preview:
        print(f"First record fields: {list(records_preview[0].keys())}")

## 3. Data Extraction
Load data from the primary record set (referenced by its `@id`) into a DataFrame for further analysis.

**Note:** For this dataset, the Croissant schema may provide a single main record set for the primary table of clinicopathological records. If multiple record sets are declared, use their `@id`. Otherwise, the records can be read without specifying `record_set`. All field/column references below must use their `@id`.

In [ ]:
# If record_sets are defined, collect their IDs; otherwise, use the default (unnamed) tabular resource.
record_set_ids = [rs.id for rs in dataset.record_sets] if dataset.record_sets else [None]
dataframes = {}

for record_set_id in record_set_ids:
    if record_set_id is None:
        # Load records without specifying record set
        records = list(dataset.records())
    else:
        records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id or 'default'] = pd.DataFrame(records)

# Preview columns from the primary table
main_df_key = record_set_ids[0] if record_set_ids[0] is not None else 'default'
print("Columns available in main record set:")
print(dataframes[main_df_key].columns.tolist())
dataframes[main_df_key].head()

## 4. Exploratory Data Analysis (EDA)

Here we apply some common data processing steps to the main clinicopathological table, including:

* Filtering records by a numeric field (e.g., patient age or diagnosis interval)
* Normalizing a numeric field (z-score)
* Grouping records by a categorical attribute (e.g., sex, MSI status)

**Reminder:** Always use the exact `@id` for record set, field, or column selection.

In [ ]:
# Choose a candidate numeric field and group field: inspect column names
main_df = dataframes[main_df_key]
print("Available columns (use @id as needed):", main_df.columns.tolist())

# For demonstration, let's guess likely field IDs from column names:
candidate_numeric_fields = [c for c in main_df.columns if any(x in c.lower() for x in ['age', 'interval', 'duration', 'years'])]
print('Numeric field candidates:', candidate_numeric_fields)

# Suppose we find e.g. '@id:diagnosis_interval_years' for the interval between cancers, and '@id:sex' and '@id:msi_status' as potential group fields
numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else main_df.columns[0]

# Filter: keep records where value > threshold
threshold = 2  # Example: 2 years
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Choose a grouping field, such as 'sex' or 'msi_status':
candidate_group_fields = [c for c in main_df.columns if any(k in c.lower() for k in ['sex', 'msi', 'subtype', 'anatomy', 'location'])]
print('\nCategorical (group) field candidates:', candidate_group_fields)
group_field = candidate_group_fields[0] if candidate_group_fields else None

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df)
else:
    print("No suitable group field found in the columns.")

## 5. Visualization
Visualize the distribution or relationship between fields using basic plots. Plots below use column `@id`s for explicit referencing.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(6,4))
sns.histplot(main_df[numeric_field], kde=True, color='teal')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# Boxplot of numeric field grouped by group_field
if group_field and group_field in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=main_df[group_field], y=main_df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
- The FAIR² dataset provides rich clinical, pathological, and molecular annotations regarding colorectal cancer in cancer survivors.
- Key fields can be referenced and processed by their `@id`, supporting robust data wrangling and reproducible selection.
- Filtering, normalization, grouping, and visualization enable insights into biomarker prevalence, anatomical risks, and patient stratification.
- For all further analyses, always refer to fields by their Croissant `@id` to maintain interoperability and standardization.